# Bolt YOLOv8 Segmentation — Fast Upload

`bolt_seg_colab.zip` 하나만 업로드합니다. 먼저 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

In [ ]:
!pip install -q ultralytics==8.4.86

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
ZIP_PATH = Path('/content/bolt_seg_colab.zip')
assert ZIP_PATH.is_file(), f'파일 없음: {ZIP_PATH}'
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall('/content')
DATASET_DIR = Path('/content/bolt_seg')
RUNS_DIR = Path('/content/runs')
print('dataset ready:', DATASET_DIR)

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n-seg.pt')
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=40,
    patience=10,
    imgsz=512,
    batch=16,
    device=device,
    workers=2,
    cache=True,
    project=str(RUNS_DIR),
    name='bolt_seg_fast',
)

In [ ]:
best_path = Path(results.save_dir) / 'weights' / 'best.pt'
print('downloading:', best_path)
files.download(str(best_path))